# 02 圖片生成（Z-Image）

**執行階段 > 全部執行**：掛載 Drive → 啟動 ComfyUI → 用下面表單的內容生成一張圖 → PNG 存到 `AI-Workflow/outputs/images/`。

要再生一張：改表單，只重跑第 3 格。需要 **L4** 以上的 GPU。第一次執行會下載模型（約 21 GB）並存進工作區，之後的 session 直接從工作區讀取。

安全負面詞與 CFG 下限（1.5）由 Core 套用，表單無法關閉。

In [ ]:
#@title 1. 掛載 Google Drive，載入 Core { display-mode: "form" }
ROOT = "/content/drive/MyDrive/AI-Workflow"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import sys
sys.path.insert(0, ROOT)
from controller import notebook as nb
session = nb.connect(ROOT, mount=False)

In [ ]:
#@title 2. 啟動 ComfyUI { display-mode: "form" }
stats = nb.start_comfyui(session)
nb.check_models(session, "z-image-basic")

In [ ]:
#@title 3. 生成圖片 { display-mode: "form" }
#@markdown ### Workflow
workflow = "z-image-basic"  #@param ["z-image-basic", "test-generation"] {allow-input: true}
#@markdown ### Simple
#@markdown Prompt（原樣送給模型，中英文都可以）
prompt = "A white ceramic mug on a plain white background, centered, soft studio light."  #@param {type:"string"}
#@markdown 比例
aspect = "1:1"  #@param ["1:1", "16:9", "9:16", "4:3", "3:4", "3:2", "2:3"]
#@markdown ---
#@markdown ### Advanced（打勾後，下面的欄位才會生效；0 或空白 = 用預設值）
use_advanced = False  #@param {type:"boolean"}
#@markdown Negative Prompt（接在安全負面詞後面，不會取代它）
negative = ""  #@param {type:"string"}
#@markdown 寬（自訂尺寸，與比例擇一）
width = 0  #@param {type:"integer"}
#@markdown 高
height = 0  #@param {type:"integer"}
#@markdown Steps
steps = 8  #@param {type:"integer"}
#@markdown CFG（最低 1.5）
cfg = 2.0  #@param {type:"number"}
#@markdown Seed（-1 = 隨機）
seed = -1  #@param {type:"integer"}
#@markdown 畫面只有一個人（加上排除第二個人的負面詞）
solo = False  #@param {type:"boolean"}
#@markdown 允許畫面出現文字
allow_text = False  #@param {type:"boolean"}
#@markdown 其他參數（JSON，例如 `{"steps": 12}`；可用的名稱見 `nb.describe(session, workflow)`）
extra_json = "{}"  #@param {type:"string"}
simple = dict(prompt=prompt, aspect=aspect)
advanced = dict(negative=negative, width=width, height=height, steps=steps, cfg=cfg, seed=seed, solo=solo, allow_text=allow_text) if use_advanced else None
result = nb.generate(session, workflow, simple, advanced, extra=extra_json if use_advanced else None)

In [ ]:
#@title 4. （選用）繼續等待 Agent 的 job { display-mode: "form" }
#@markdown 勾選後，這本 notebook 會像 01 一樣留著執行佇列裡的 job。
wait_for_jobs = False  #@param {type:"boolean"}
#@markdown 佇列空了這麼多分鐘後停止等待：
idle_minutes = 10  #@param {type:"integer"}
#@markdown 停止後釋放 runtime（不再消耗運算單元）。取消勾選的話 runtime 會繼續計費，要自己中斷。
release_runtime = True  #@param {type:"boolean"}
if wait_for_jobs:
    summary = nb.serve(session, idle_minutes, release_runtime)